# Weights configuration for Jaccard Index generation

In this notebook, I will be doing experimentation, training of models, and evaluation of different weight configurations for generating the Jaccard Index.


In [1]:
import psycopg
import pandas as pd
import numpy as np
import igraph as ig
from sshtunnel import SSHTunnelForwarder
import dotenv
import os
dotenv.load_dotenv()

True

In [26]:
server = SSHTunnelForwarder(
    os.getenv("SSH_HOST"),
    ssh_username=os.getenv("SSH_USERNAME"),
    ssh_pkey=os.getenv("SSH_PKEY_PATH"),
    remote_bind_address=('127.0.0.1', 5432)
)
server.start()
conn = psycopg.connect(f"dbname={os.getenv('DB_NAME')} user=postgres host=127.0.0.1 port={server.local_bind_port} password={os.getenv('DB_PASSWORD')}")
conn.read_only = True
cur = conn.cursor()


In [36]:
START_DATE = '2026-08-27'
END_DATE = '2026-09-01'

In [27]:
cur.execute('SELECT * FROM part_types;')
print([desc[0] for desc in cur.description])
part_types = cur.fetchall()
print(part_types)

['id', 'name', 'description']
[(UUID('07901ba7-64d3-4603-9f5c-38a761ea9ed6'), 'method', 'HTTP Method (GET, POST, etc.)'), (UUID('e1b77c40-928f-4a87-9130-5ce6aeb91b62'), 'host', 'The host of the request'), (UUID('d3b24ec8-85e9-414a-9639-0d4b489388d9'), 'query_part', 'Part of the query part of the URL'), (UUID('6ff317f1-d8ec-48f0-8c6a-5e146c799185'), 'path_part', 'Part of the endpoint path of the URL'), (UUID('a515a5f3-4354-4186-b264-f3b616ab8197'), 'body_part', 'Part of the body of the request'), (UUID('eaec0218-0f1d-487b-9b63-4543a6422e36'), 'header', 'HTTP Header (User-Agent, Content-Type, etc.)'), (UUID('2165835b-b1b1-4da3-bbf6-683d0e82856d'), 'user_agent', 'The user agent of the request'), (UUID('756a9262-c513-4cc4-9f3f-3730f4055d35'), 'request_length', 'Length of the HTTP request'), (UUID('d8894b05-3053-4c45-9f58-03ebb3ee2f15'), 'status_code', 'HTTP Status Code'), (UUID('8ac61f8d-81b3-4e29-bb9e-13bfb22974ed'), 'server_protocol', 'The protocol used by the server (HTTP/1.1, HTTP/2, e

In [ ]:
query = """
WITH selectedRequests AS (
  SELECT r.id AS request_id
  FROM requests r
  WHERE r.created_at >= %s::date AND r.created_at < %s::date
), part_weights(name, weight) AS (
    VALUES
        ('method', 1),
        ('host', 1),
        ('query_part', 1),
        ('path_part', 1),
        ('body_part', 1),
        ('header', 1),
        ('user_agent', 1),
        ('request_length', 1),
        ('status_code', 1),
        ('server_protocol', 1),
        ('server_name', 1),
        ('remote_addr', 1),
        ('accept_encoding', 1),
        ('content_type', 1),
        ('scheme', 1)
), shared_parts AS (
    SELECT rp1.request_id AS request_1,
        rp2.request_id AS request_2,
        SUM(pw.weight) AS shared_parts_count
    FROM request_to_parts rp1
    JOIN request_to_parts rp2 ON rp1.part_id = rp2.part_id
    JOIN request_parts p ON rp1.part_id = p.id
    JOIN part_types pt ON p.part_type = pt.id
    JOIN part_weights pw ON pt.name = pw.name
    WHERE p.excluded = FALSE
      AND rp1.request_id < rp2.request_id
      AND rp1.request_id IN (SELECT request_id FROM selectedRequests)
      AND rp2.request_id IN (SELECT request_id FROM selectedRequests)
    GROUP BY rp1.request_id, rp2.request_id
),
non_excluded_parts AS (
    SELECT r.id AS request_id, SUM(pw.weight) AS non_excluded_count
    FROM requests r
    JOIN request_to_parts rtp ON r.id = rtp.request_id
    JOIN request_parts rp ON rtp.part_id = rp.id
    JOIN part_types pt ON rp.part_type = pt.id
    JOIN part_weights pw ON pt.name = pw.name
    WHERE rp.excluded = FALSE
      AND r.id IN (SELECT request_id FROM selectedRequests)
    GROUP BY r.id
),
full_graph AS (
    SELECT r1.ext_id AS request_1,
        r2.ext_id AS request_2,
        sp.shared_parts_count::float / (
            nep1.non_excluded_count + nep2.non_excluded_count - sp.shared_parts_count
        )::float AS jaccard_index
    FROM shared_parts sp
    JOIN requests r1 ON sp.request_1 = r1.id
    JOIN requests r2 ON sp.request_2 = r2.id
    JOIN non_excluded_parts nep1 ON sp.request_1 = nep1.request_id
    JOIN non_excluded_parts nep2 ON sp.request_2 = nep2.request_id
), filtered AS (
    SELECT * FROM full_graph
    WHERE jaccard_index > (SELECT value::float FROM loggraph_config WHERE key = 'min_jaccard_index')
)
SELECT * FROM filtered;
"""
data = cur.execute(query, (START_DATE, END_DATE)).fetchall()
print(len(data))

1773820


Now we will build the graph: for each request, we will create nodes for its parts and connect them based on their relationships. Expected row count 1773820.


In [40]:
print(data[:10])

[(2054, 2090, 0.6), (153, 1632, 0.3333333333333333), (2076, 6168, 0.6), (2076, 6324, 0.3333333333333333), (61, 70, 0.3333333333333333), (2054, 6143, 0.3333333333333333), (189, 1657, 0.3333333333333333), (21, 6568, 0.3333333333333333), (6065, 5986, 0.3333333333333333), (6019, 6036, 0.6)]
